# Cell cycle scoring: hPGC in vivo vs hPGCLC in vitro

### One thing to keep in mind about G1!!!

The way the algorithm works, a G1 call here means neither the S nor the
G2M programme is on, which includes quiescent and arrested cells in with
G1 proper. If a difference is in G1 rather than in S or G2M, you need to
be careful!!!

### Three groups (not just standard two)

The in vivo side is reported twice, as all germ cells, and as the DAZL/DDX4
negative early subset, so the early cells are deliberately counted in both.
If the in vivo and in vitro cycling rates differ but the early subset sits
closer to the in vitro value, part of that difference was developmental stage
rather than culture conditions.


## 1. Settings

In [ ]:
import os
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt

# ---- inputs ---------------------------------------------------------------
INVIVO_H5AD  = "path_to_data/invivo_window_female.h5ad" #change to your directory
INVITRO_H5AD = "path_to_data/invitro_integrated_scanpy_joao.h5ad" #change to your directory

INVIVO_CELLTYPE_KEY  = "fine_label"
INVIVO_SAMPLE_KEY    = "donor_id"  
INVIVO_GERM          = "primordial germ cell"  

INVITRO_CELLTYPE_KEY = "cell_type"
INVITRO_SAMPLE_KEY   = "sample"
INVITRO_GERM         = "PGCLC" 

# ---- which matrix to score ------------------------------------------------
# Scoring needs LOG NORMALISED values. This is the opposite of what DESeq2
# wants in the DE notebook. None means use adata.X, and a copy is normalised
# on the fly if X turns out to be counts.
LOGNORM_LAYER = None                            # CHANGE IF NEEDED

# ---- analysis choices -----------------------------------------------------
MIN_CELLS  = 10      # a sample needs this many germ cells to be plotted
EARLY_MARKERS = ["DAZL", "DDX4"]   # negative for both defines the early subset

OUTDIR = "results_cellcycle"
os.makedirs(OUTDIR, exist_ok=True)

## 2. Load and subset to germ cells

In [ ]:
vivo  = sc.read_h5ad(INVIVO_H5AD)
vitro = sc.read_h5ad(INVITRO_H5AD)

for name, a, ct_key, germ in [("vivo", vivo, INVIVO_CELLTYPE_KEY, INVIVO_GERM),
                              ("vitro", vitro, INVITRO_CELLTYPE_KEY, INVITRO_GERM)]:
    a.obs[ct_key] = a.obs[ct_key].astype(str)
    n = int((a.obs[ct_key] == germ).sum())
    if n == 0:
        raise ValueError(f"{name}: no cells labelled {germ!r} under {ct_key!r}. "
                         f"Present: {sorted(a.obs[ct_key].unique())[:25]}")
    print(f"{name}: {a.n_obs} cells, {n} labelled {germ!r}")

germ_vivo  = vivo[vivo.obs[INVIVO_CELLTYPE_KEY] == INVIVO_GERM].copy()
germ_vitro = vitro[vitro.obs[INVITRO_CELLTYPE_KEY] == INVITRO_GERM].copy()

germ_vivo.obs["sample_id"]  = germ_vivo.obs[INVIVO_SAMPLE_KEY].astype(str)
germ_vitro.obs["sample_id"] = germ_vitro.obs[INVITRO_SAMPLE_KEY].astype(str)

# Mark the early subset on the in vivo side. NOT A FILTER: it becomes a third
# reporting group so the stage effect is visible in the same run.
late = np.zeros(germ_vivo.n_obs, dtype=bool)
for g in EARLY_MARKERS:
    if g in germ_vivo.var_names:
        v = germ_vivo[:, g].X
        v = v.toarray().ravel() if hasattr(v, "toarray") else np.asarray(v).ravel()
        late |= (v > 0)
    else:
        print(f"  WARNING: {g} not in var_names, the early subset ignores it")
germ_vivo.obs["stage"] = np.where(late, "later", "early")
print(f"\nin vivo germ cells: {int((~late).sum())} early, {int(late.sum())} later")

print(f"independent samples: {germ_vivo.obs['sample_id'].nunique()} in vivo, "
      f"{germ_vitro.obs['sample_id'].nunique()} in vitro")

## 3. Check if genes match

Same as in DE.

In [ ]:
shared_genes = sorted(set(germ_vivo.var_names) & set(germ_vitro.var_names))
print(f"{len(germ_vivo.var_names)} vivo genes, {len(germ_vitro.var_names)} vitro genes, "
      f"{len(shared_genes)} shared")
if len(shared_genes) < 5000:
    raise ValueError("Fewer than 5000 shared genes. The two objects are probably "
                     "using different identifier types (Ensembl versus symbol).")

germ_vivo  = germ_vivo[:, shared_genes].copy()
germ_vitro = germ_vitro[:, shared_genes].copy()

## 4. Marker genes

The Tirosh et al. 2015 S phase and G2/M sets, as used by Seurat and scanpy.
Three symbols were renamed after that list was published, so aliases are handled
rather than silently dropped.

In [ ]:
S_GENES = """MCM5 PCNA TYMS FEN1 MCM2 MCM4 RRM1 UNG GINS2 MCM6 CDCA7 DTL PRIM1
UHRF1 MLF1IP HELLS RFC2 RPA2 NASP RAD51AP1 GMNN WDR76 SLBP CCNE2 UBR7 POLD3 MSH2
ATAD2 RAD51 RRM2 CDC45 CDC6 EXO1 TIPIN DSCC1 BLM CASP8AP2 USP1 CLSPN POLA1 CHAF1B
BRIP1 E2F8""".split()

G2M_GENES = """HMGB2 CDK1 NUSAP1 UBE2C BIRC5 TPX2 TOP2A NDC80 CKS2 NUF2 CKS1B
MKI67 TMPO CENPF TACC3 FAM64A SMC4 CCNB2 CKAP2L CKAP2 AURKB BUB1 KIF11 ANP32E
TUBB4B GTSE1 KIF20B HJURP CDCA3 HN1 CDC20 TTK CDC25C KIF2C RANGAP1 NCAPD2 DLGAP5
CDCA2 CDCA8 ECT2 KIF23 HMMR AURKA PSRC1 ANLN LBR CKAP5 CENPE CTCF NEK2 G2E3
GAS2L3 CBX5 CENPA""".split()

ALIASES = {"MLF1IP": "CENPU", "FAM64A": "PIMREG", "HN1": "JPT1"}

def resolve(genes, present):
    out, renamed, missing = [], [], []
    for g in genes:
        if g in present:
            out.append(g)
        elif g in ALIASES and ALIASES[g] in present:
            out.append(ALIASES[g]); renamed.append(f"{g}->{ALIASES[g]}")
        else:
            missing.append(g)
    return out, renamed, missing

present = set(shared_genes)
s_genes,   s_ren, s_mis = resolve(S_GENES, present)
g2m_genes, g_ren, g_mis = resolve(G2M_GENES, present)

print(f"S phase: {len(s_genes)} of {len(S_GENES)} found")
print(f"G2M:     {len(g2m_genes)} of {len(G2M_GENES)} found")
if s_ren or g_ren:
    print(f"resolved via alias: {', '.join(s_ren + g_ren)}")
if s_mis or g_mis:
    print(f"not found: {', '.join(s_mis + g_mis)}")
if min(len(s_genes), len(g2m_genes)) < 20:
    print("WARNING: few markers recovered, the phase calls will be unreliable")

## 5. Score

Each cell gets an S score and a G2M score, and is assigned the phase whose score
is highest, or G1 if neither is positive. Mentioned in intro, be careful with what
G1 really is!

In [ ]:
def looks_like_counts(X, n=200):
    head = X[:n]
    head = head.toarray() if hasattr(head, "toarray") else np.asarray(head)
    return np.allclose(head, np.round(head)), float(head.max())

def score_cycle(adata, name):
    a = adata.copy()
    if LOGNORM_LAYER:
        a.X = a.layers[LOGNORM_LAYER].copy()
    is_int, mx = looks_like_counts(a.X)
    print(f"  {name}: matrix max {mx:.2f}, integer valued {is_int}")
    if is_int and mx > 30:
        print(f"    looks like raw counts, normalising a copy for scoring")
        sc.pp.normalize_total(a, target_sum=1e4)
        sc.pp.log1p(a)
    sc.tl.score_genes_cell_cycle(a, s_genes=s_genes, g2m_genes=g2m_genes)
    for col in ["S_score", "G2M_score", "phase"]:
        adata.obs[col] = a.obs[col].values
    return adata

germ_vivo  = score_cycle(germ_vivo,  "vivo")
germ_vitro = score_cycle(germ_vitro, "vitro")

# One long table. The barcode is kept as a column and the index is made unique
# by prefixing condition and sample, because 10x barcodes routinely collide
# between two independently processed objects.
v = germ_vivo.obs[["sample_id", "stage", "phase", "S_score", "G2M_score"]].copy()
v["cell_id"] = germ_vivo.obs_names
v["condition"] = "in vivo"
v["group"] = np.where(v["stage"] == "early", "in vivo (early)", "in vivo (later)")

t = germ_vitro.obs[["sample_id", "phase", "S_score", "G2M_score"]].copy()
t["cell_id"] = germ_vitro.obs_names
t["condition"], t["stage"], t["group"] = "in vitro", "n/a", "in vitro"

cc = pd.concat([v, t], ignore_index=True)
cc["phase"] = pd.Categorical(cc["phase"], categories=["G1", "S", "G2M"])
cc.index = (cc["condition"].str.replace(" ", "_") + "|" + cc["sample_id"]
            + "|" + cc["cell_id"])
if not cc.index.is_unique:
    print("WARNING: duplicate cell identifiers even after prefixing, check sample keys")

GROUPS = ["in vivo (all)", "in vivo (early)", "in vitro"]
print(f"\nscored {len(cc)} cells")
print(cc.groupby("group", observed=True).size().to_string())

## 6. Phase percentages

The first table pools all cells and is descriptive only. The second computes
percentages within each sample and summarises across samples.

In [ ]:
# Build the three reporting groups. "in vivo (all)" includes early plus later, so
# the early cells appear in two groups and ignore_index is required: crosstab
# cannot reindex an axis carrying duplicate labels.
frames = []
for label, sel in [("in vivo (all)",   cc["condition"] == "in vivo"),
                   ("in vivo (early)", cc["group"] == "in vivo (early)"),
                   ("in vitro",        cc["group"] == "in vitro")]:
    d = cc[sel].copy(); d["_grp"] = label
    frames.append(d)
allg = pd.concat(frames, ignore_index=True)

pooled = (pd.crosstab(allg["_grp"], allg["phase"], normalize="index") * 100).reindex(GROUPS)
pooled["n_cells"] = allg.groupby("_grp", observed=True).size().reindex(GROUPS)
for ph in ["G1", "S", "G2M"]:
    if ph not in pooled:
        pooled[ph] = 0.0
print("Pooled across cells (descriptive only):")
print(pooled[["G1", "S", "G2M", "n_cells"]].round(1).to_string())

# n_cells is joined on the shared MultiIndex rather than assigned by position,
# which would silently misalign if the two orderings ever diverged.
pct = (pd.crosstab([allg["_grp"], allg["sample_id"]], allg["phase"],
                   normalize="index") * 100)
n = allg.groupby(["_grp", "sample_id"], observed=True).size().rename("n_cells")
per_sample = pct.join(n).reset_index().rename(columns={"_grp": "group"})
for ph in ["G1", "S", "G2M"]:
    if ph not in per_sample:
        per_sample[ph] = 0.0
per_sample["cycling"] = per_sample["S"] + per_sample["G2M"]
small = per_sample[per_sample["n_cells"] < MIN_CELLS]
if len(small):
    print(f"\n{len(small)} sample-group combinations under MIN_CELLS, excluded from "
          f"the summary below:")
    print(small[["group", "sample_id", "n_cells"]].to_string(index=False))
per_sample = per_sample[per_sample["n_cells"] >= MIN_CELLS]

print("\nPer sample:")
print(per_sample.round(1).to_string(index=False))

summary = (per_sample.groupby("group", observed=True)[["G1", "S", "G2M", "cycling"]]
           .agg(["mean", "std", "min", "max"]).round(1).reindex(GROUPS))
print("\nAcross samples, mean and spread of the per sample percentages:")
print(summary.to_string())

### Comparing the cycling fraction

Two comparisons against the in vitro cells: all in vivo germ cells, and the
early subset only. If the early comparison is the smaller difference, part of
what separates the two systems was developmental stage rather than culture.

In [ ]:
from scipy.stats import mannwhitneyu

def cyc(label):
    return per_sample.loc[per_sample.group == label, "cycling"]

vt = cyc("in vitro")
for label in ["in vivo (all)", "in vivo (early)"]:
    vv = cyc(label)
    print(f"{label:18s} {vv.mean():5.1f}% (n={len(vv)} samples)   vs   "
          f"in vitro {vt.mean():5.1f}% (n={len(vt)})   "
          f"difference {vt.mean() - vv.mean():+.1f} points")
    if len(vv) >= 3 and len(vt) >= 3:
        u, p = mannwhitneyu(vv, vt, alternative="two-sided")
        print(f"{'':18s} Mann-Whitney on per sample percentages: p = {p:.4f}")
    else:
        print(f"{'':18s} too few samples for a test, read the spread instead")

print("\nThis tests samples, not cells. With only a few samples per side the "
      "smallest achievable p value is limited.")

### Plots

In [ ]:
COLP = {
    "G1": "#6A94CE",    # process cyan
    "S": "#D2588D",     # process magenta
    "G2M": "#F5D751",   # process yellow
}

fig = plt.figure(figsize=(16, 9))
gs = fig.add_gridspec(2, 2, width_ratios=[1, 1.6], height_ratios=[1, 1])

# ---- pooled, three groups -------------------------------------------------
ax = fig.add_subplot(gs[0, 0])
bottom = np.zeros(len(GROUPS))
for ph in ["G1", "S", "G2M"]:
    vals = pooled[ph].values
    ax.bar(GROUPS, vals, bottom=bottom, color=COLP[ph], label=ph, width=0.6)
    for i, v in enumerate(vals):
        if v > 5:
            ax.text(i, bottom[i] + v / 2, f"{v:.0f}%", ha="center", va="center", fontsize=9)
    bottom += vals
ax.set_ylabel("percent of cells")
ax.set_title("Pooled (descriptive)")
ax.tick_params(axis="x", labelrotation=20)
ax.legend(frameon=False, fontsize=8)

# ---- per sample -----------------------------------------------------------
ax = fig.add_subplot(gs[0, 1])
ps = per_sample.sort_values(["group", "sample_id"])
x = np.arange(len(ps)); bottom = np.zeros(len(ps))
for ph in ["G1", "S", "G2M"]:
    ax.bar(x, ps[ph].values, bottom=bottom, color=COLP[ph], width=0.75)
    bottom += ps[ph].values
edges = np.cumsum(ps.groupby("group", observed=True, sort=False).size().values)[:-1]
for e in edges:
    ax.axvline(e - 0.5, color="black", lw=1.2)
ax.set_xticks(x)
ax.set_xticklabels([f"{r.group.split()[-1].strip('()')} {r.sample_id}"
                    for r in ps.itertuples()], rotation=70, ha="right", fontsize=7)
ax.set_ylabel("percent of cells")
ax.set_title("Per sample, groups separated by lines")

# ---- score scatter, one panel per group -----------------------------------
for j, label in enumerate(["in vivo (early)", "in vitro"]):
    ax = fig.add_subplot(gs[1, j])
    d = allg[allg["_grp"] == label]
    for ph in ["G1", "S", "G2M"]:
        m = (d["phase"] == ph).values
        ax.scatter(d.loc[m, "S_score"], d.loc[m, "G2M_score"], s=10,
                   c=COLP[ph], label=ph, alpha=0.7, linewidths=0)
    ax.axhline(0, lw=0.6, c="grey"); ax.axvline(0, lw=0.6, c="grey")
    ax.set_xlabel("S score"); ax.set_ylabel("G2M score")
    ax.set_title(f"{label}, n={len(d)}")
    ax.legend(frameon=False, fontsize=8, markerscale=1.6)

plt.tight_layout()
#plt.savefig(os.path.join(OUTDIR, "cellcycle_composition.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
from scipy.stats import mannwhitneyu

# restrict to cells belonging to samples that passed the MIN_CELLS filter
keep_index = pd.MultiIndex.from_frame(per_sample[["group", "sample_id"]])
allg_index = pd.MultiIndex.from_arrays([allg["_grp"], allg["sample_id"]])
allg_f = allg[allg_index.isin(keep_index)].copy()

pooled_f = (pd.crosstab(allg_f["_grp"], allg_f["phase"], normalize="index") * 100).reindex(GROUPS)
for ph in ["G1", "S", "G2M"]:
    if ph not in pooled_f:
        pooled_f[ph] = 0.0
        
selected_groups = ["in vivo (all)", "in vitro"]
display_labels = {"in vivo (all)": "in vivo", "in vitro": "in vitro"}

indices = [GROUPS.index(g) for g in selected_groups]

fig, axes = plt.subplots(1, len(selected_groups), figsize=(4 * len(selected_groups), 4))
if len(selected_groups) == 1:
    axes = [axes]

phase_vals = {ph: pooled_f[ph].values for ph in ["G1", "S", "G2M"]}

for ax, grp, idx in zip(axes, selected_groups, indices):
    vals = [phase_vals[ph][idx] for ph in ["G1", "S", "G2M"]]
    n_cells = len(allg_f[allg_f["_grp"] == grp])
    n_samples = len(per_sample.loc[per_sample.group == grp])
    ax.pie(
        vals,
        colors=[COLP[ph] for ph in ["G1", "S", "G2M"]],
        autopct=lambda p: f"{p:.0f}%" if p > 5 else "",
        startangle=90,
    )
    ax.set_title(f"{display_labels[grp]} (n={n_cells} cells, {n_samples} samples)", fontsize=10)

# Mann-Whitney on per-sample % cycling: in vivo (all) vs in vitro
vt = per_sample.loc[per_sample.group == "in vitro", "cycling"]
vv = per_sample.loc[per_sample.group == "in vivo (all)", "cycling"]
if len(vv) >= 3 and len(vt) >= 3:
    u, p = mannwhitneyu(vv, vt, alternative="two-sided")
    stat_text = f"Mann-Whitney (per-sample % cycling): p = {p:.4f}"
else:
    stat_text = "Mann-Whitney: too few samples for a test"

fig.text(0.5, 0.90, stat_text, ha="center", fontsize=9, style="italic")

fig.legend(
    [plt.Rectangle((0, 0), 1, 1, color=COLP[ph]) for ph in ["G1", "S", "G2M"]],
    ["G1", "S", "G2M"],
    loc="lower center", ncol=3, frameon=False, fontsize=9,
)
fig.suptitle("Cell Cycle Composition (Pooled)")
plt.tight_layout(rect=[0, 0.05, 1, 0.85])
plt.savefig(os.path.join(OUTDIR, "cellcycle_pooled_pie.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ps = per_sample.sort_values(["group", "sample_id"])
x = np.arange(len(ps)); bottom = np.zeros(len(ps))
for ph in ["G1", "S", "G2M"]:
    ax.bar(x, ps[ph].values, bottom=bottom, color=COLP[ph], width=0.75, label=ph)
    bottom += ps[ph].values
edges = np.cumsum(ps.groupby("group", observed=True, sort=False).size().values)[:-1]
for e in edges:
    ax.axvline(e - 0.5, color="black", lw=1.2)
ax.set_xticks(x)
ax.set_xticklabels(
    [f"{r.group.split()[-1].strip('()')} {r.sample_id}\n(n={int(r.n_cells)})"
     for r in ps.itertuples()],
    rotation=45, ha="right", fontsize=7,
)
ax.set_ylabel("percent of cells")
ax.set_title("Per sample (in vitro, in vivo, in vivo (early))")
ax.legend(frameon=False, fontsize=8, loc='center left', bbox_to_anchor=(1.02, 0.5))
plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, "cellcycle_per_sample.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
for label in ["in vivo (all)", "in vivo (early)", "in vitro"]:
    fig, ax = plt.subplots(figsize=(6, 6))
    d = allg[allg["_grp"] == label]
    for ph in ["G1", "S", "G2M"]:
        m = (d["phase"] == ph).values
        ax.scatter(d.loc[m, "S_score"], d.loc[m, "G2M_score"], s=10,
                   c=COLP[ph], label=ph, alpha=0.7, linewidths=0)
    ax.axhline(0, lw=0.6, c="grey"); ax.axvline(0, lw=0.6, c="grey")
    ax.set_xlabel("S score"); ax.set_ylabel("G2M score")
    ax.set_title(f"{label}, n={len(d)}")
    ax.legend(frameon=False, fontsize=8, markerscale=1.6)
    plt.tight_layout()
    fname = label.replace(" ", "_").replace("(", "").replace(")", "")
    plt.savefig(os.path.join(OUTDIR, f"cellcycle_scatter_{fname}.png"), dpi=150, bbox_inches="tight")
    plt.show()

## 7. Export

`cellcycle_per_cell.csv` is indexed by cell barcode, so the DE notebook can read
it back if needed and merge on the index rather than re-running any of this.

In [ ]:
cc.to_csv(os.path.join(OUTDIR, "cellcycle_per_cell.csv"))
per_sample.to_csv(os.path.join(OUTDIR, "cellcycle_per_sample.csv"), index=False)
pooled.to_csv(os.path.join(OUTDIR, "cellcycle_pooled.csv"))
print("wrote to", OUTDIR)

# To use these in the pseudobulk DE notebook, after building germ_vivo/germ_vitro:
#
#   cc = pd.read_csv("results_cellcycle/cellcycle_per_cell.csv", index_col=0)
#   vv = cc[cc.condition == "in vivo"].set_index("cell_id")
#   vt = cc[cc.condition == "in vitro"].set_index("cell_id")
#   germ_vivo.obs["phase"]  = vv["phase"].reindex(germ_vivo.obs_names).values
#   germ_vitro.obs["phase"] = vt["phase"].reindex(germ_vitro.obs_names).values
#
# Split by condition first: barcodes can repeat across the two objects, so
# reindexing the combined table would raise or match the wrong cells.
#
# then flag cell cycle genes in the results table so they can be discounted:
#
#   res["cell_cycle"] = res.gene.isin(set(s_genes) | set(g2m_genes))
print("\nS and G2M gene lists, for flagging DE hits:")
print(f"  {len(s_genes)} S genes, {len(g2m_genes)} G2M genes, "
      f"{len(set(s_genes) | set(g2m_genes))} total")
pd.Series(sorted(set(s_genes) | set(g2m_genes)), name="gene").to_csv(
    os.path.join(OUTDIR, "cellcycle_genes_used.csv"), index=False)

## Notes

**Check the early comparison against the all comparison.** If the difference
shrinks when the in vivo side is restricted to early PGCs, then developmental
stage was part of it, and the same will be true of the DE.

**Small groups give noisy phase calls.** A 36 cell early subset gives
percentages in steps of about 2.8 points, so treat them as indicative.